In [1]:
import pandas as pd
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from zero_point import zpt
zpt.load_tables()

In [2]:
import sys
from pathlib import Path

# Ajustar el path del proyecto para importar el módulo desde src/
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "src").exists():
    project_root = project_root.parent
project_root

WindowsPath('c:/Users/nicob/One Drive Uniandes/OneDrive - Universidad de los Andes/Doctorado/proyecto/clusterization_project')

In [3]:

# df = pd.read_csv(project_root / "data" / "datos_resultados_modularizado" / "datos_clusterizados_todos_5d_f00.csv")
data_path = project_root / "data" / "datos_simulados" / "mock_open_cluster_gaia_hyades_clean.csv"
df = pd.read_csv(data_path)

In [5]:
# Funciones vectorizadas para correccion de paralaje y movimiento propio
# Usando operaciones de numpy para máximo rendimiento

def get_rotation_vectorized(G):
    """
    Calcula wx, wy, wz de forma vectorizada para arrays de magnitudes.
    
    Parameters
    ----------
    G : np.ndarray
        Array de magnitudes G
        
    Returns
    -------
    tuple of np.ndarray
        (wx, wy, wz) arrays
    """
    wx = np.full_like(G, 0, dtype=np.float64)
    wy = np.full_like(G, 0, dtype=np.float64)
    wz = np.full_like(G, 0, dtype=np.float64)
    
    mask1 = G < 9
    wx[mask1], wy[mask1], wz[mask1] = -5, -3, 0
    
    mask2 = (G >= 9) & (G < 11)
    wx[mask2], wy[mask2], wz[mask2] = -10, -5, 2
    
    mask3 = (G >= 11) & (G < 13)
    wx[mask3], wy[mask3], wz[mask3] = -25, -15, 5
    
    return wx, wy, wz

def pm_correction_vectorized(mu_ra, mu_dec, ra, dec, wx, wy, wz):
    """
    Corrige movimiento propio de forma vectorizada usando operaciones numpy.
    
    Parameters
    ----------
    mu_ra, mu_dec : np.ndarray
        Movimientos propios en RA y Dec (mas/yr)
    ra, dec : np.ndarray
        Coordenadas ecuatoriales (grados)
    wx, wy, wz : np.ndarray
        Parámetros de rotación del sistema de referencia
        
    Returns
    -------
    tuple of np.ndarray
        (mu_ra_corr, mu_dec_corr) corregidos
    """
    # Conversión a radianes (vectorizado)
    ra_rad = np.deg2rad(ra)
    dec_rad = np.deg2rad(dec)
    
    # Pre-computar funciones trigonométricas
    sin_ra = np.sin(ra_rad)
    cos_ra = np.cos(ra_rad)
    sin_dec = np.sin(dec_rad)
    cos_dec = np.cos(dec_rad)
    
    # Correcciones (operaciones vectorizadas)
    dmu_ra = -wx * sin_ra + wy * cos_ra
    
    dmu_dec = (-wx * cos_ra * sin_dec 
               - wy * sin_ra * sin_dec 
               + wz * cos_dec)
    
    # Aplicar correcciones
    mu_ra_corr = mu_ra - dmu_ra
    mu_dec_corr = mu_dec - dmu_dec
    
    return mu_ra_corr, mu_dec_corr


In [23]:
df_gaia = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_shell\gaia_230.csv")
df_sp = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\DatosTotales\all_fidelity.csv")

In [25]:
# aplicando correccion de paralaje
df_gaia["zp"] = zpt.get_zpt(
    df_gaia["phot_g_mean_mag"],
    df_gaia["nu_eff_used_in_astrometry"],
    df_gaia["pseudocolour"],
    df_gaia["ecl_lat"],
    df_gaia["astrometric_params_solved"]
)

c:\Users\nicob\anaconda3\Lib\site-packages\zero_point\zpt.py:215: UserWarning: The apparent magnitude of one or more of the sources is outside the expected range (6-21 mag). 
                Outside this range, there is no further interpolation, thus the values at 6 or 21 are returned.
  warnings.warn(
c:\Users\nicob\anaconda3\Lib\site-packages\zero_point\zpt.py:230: UserWarning: The nu_eff_used_in_astrometry of some of the 5p source(s) is outside the expected range (1.1-1.9 
                mag). Outside this range, the zero-point calculated can be seriously wrong.
  warnings.warn(
c:\Users\nicob\anaconda3\Lib\site-packages\zero_point\zpt.py:243: UserWarning: The pseudocolour of some of the 6p source(s) is outside the expected range (1.24-1.72 mag).
                 The maximum corrections are reached already at 1.24 and 1.72
  warnings.warn(


In [26]:
df_gaia["parallax_corrected"] = df_gaia["parallax"] - df_gaia["zp"]/1000.0

In [29]:
df_gaia.head()

,source_id,ra,dec,parallax,pmra,pmdec,ruwe,phot_g_mean_mag,bp_rp,radial_velocity,...,parallax_error,visibility_periods_used,phot_bp_mean_mag,phot_rp_mean_mag,nu_eff_used_in_astrometry,pseudocolour,ecl_lat,astrometric_params_solved,zp,parallax_corrected
0,138832313879044096,46.076795,35.864350,6.694639,6.033624,-27.336710,1.043566,15.661435,2.753020,-13.066293,...,0.048872,15,17.214370,14.461350,1.260937,NaN,17.778413,31,-0.054061,6.694693
1,138944429705118336,46.736132,36.147998,4.627168,45.396808,-19.805744,1.031152,14.599747,1.872160,26.450920,...,0.023367,14,15.523865,13.651705,1.360007,NaN,17.896349,31,-0.043037,4.627211
2,138969821551607552,46.725418,36.475977,10.614968,84.123814,-46.586764,1.233279,13.714395,1.990878,53.810375,...,0.024990,13,14.714951,12.724072,1.345367,NaN,18.213029,31,-0.043869,10.615012
3,139005624398868864,46.089722,36.526846,4.764534,-8.688261,-21.951382,0.904957,18.244907,3.059847,NaN,...,0.163873,15,20.066843,17.006996,NaN,1.145332,18.409359,95,-0.047444,4.764581
4,139012835647884416,46.002892,36.718485,6.921768,14.016314,-33.248599,1.020420,17.206024,3.051111,NaN,...,0.093710,14,18.992582,15.941471,1.234228,NaN,18.612986,31,-0.049787,6.921818


In [30]:
df = df_gaia.merge(df_sp, on='source_id', how='left')

In [32]:
len(set(df_gaia.source_id).intersection(set(df_sp.source_id)))

88988

In [33]:
# Aplicar correcciones de rotacion y movimiento propio en una sola operacion vectorizada
# Esto es O(n) en lugar de O(n*m) con apply()

# Obtener arrays numpy para operaciones vectorizadas
G = df['phot_g_mean_mag'].values
ra = df['ra'].values
dec = df['dec'].values
mu_ra = df['pmra'].values
mu_dec = df['pmdec'].values

# Calcular parámetros de rotación
wx, wy, wz = get_rotation_vectorized(G)

# Aplicar correcciones de movimiento propio
mu_ra_corr, mu_dec_corr = pm_correction_vectorized(mu_ra, mu_dec, ra, dec, wx, wy, wz)

# Asignar resultados al dataframe
df['wx'] = wx
df['wy'] = wy
df['wz'] = wz
df['pmra_corrected'] = mu_ra_corr
df['pmdec_corrected'] = mu_dec_corr


In [6]:
# Constante de conversión:
KAPPA = 4.74047

def correct_gaia_proper_motions(
    df: pd.DataFrame,
    ra_col: str = "ra",
    dec_col: str = "dec",
    parallax_col: str = "parallax",
    pmra_col: str = "pmra",
    pmdec_col: str = "pmdec",
    solar_motion_uvw_kms: tuple[float, float, float] = (11.1, 12.24, 7.25),
    correct_solar_reflex: bool = True,
    correct_oort_rotation: bool = False,
    suffix: str = "_corr",
    copy: bool = True,
) -> pd.DataFrame:
    """
    Corrige movimientos propios de Gaia.

    Esta función toma las columnas astrométricas básicas de Gaia:

        ra        [grados]
        dec       [grados]
        parallax  [mas]
        pmra      [mas/yr] = mu_alpha* = mu_alpha cos(dec)
        pmdec     [mas/yr]

    y calcula movimientos propios corregidos por:

        1. Movimiento solar reflejo respecto al Local Standard of Rest, LSR.

    La corrección se hace estrella por estrella, porque la proyección del
    movimiento solar y de la rotación galáctica depende de la posición en el cielo
    y, en el caso del movimiento solar reflejo, también de la distancia.

    Parámetros
    ----------
    df : pandas.DataFrame
        Tabla con los datos Gaia.

    ra_col, dec_col : str
        Nombres de las columnas de ascensión recta y declinación en grados.

    parallax_col : str
        Nombre de la columna de paralaje en milisegundos de arco, mas.

    pmra_col, pmdec_col : str
        Nombres de las columnas de movimiento propio en ICRS.
        En Gaia, pmra ya incluye el factor cos(dec):
            pmra = mu_alpha* = mu_alpha cos(dec)

    solar_motion_uvw_kms : tuple
        Movimiento solar respecto al LSR en km/s.

        Convención galáctica estándar:
            U > 0 hacia el centro galáctico
            V > 0 en la dirección de rotación galáctica
            W > 0 hacia el polo norte galáctico

        Valor por defecto:
            (U, V, W) = (11.1, 12.24, 7.25) km/s

    correct_solar_reflex : bool
        Si True, resta el movimiento solar reflejo de los movimientos propios.

    suffix : str
        Sufijo para las columnas corregidas.
        Por defecto se crearán:
            pmra_corr
            pmdec_corr

    copy : bool
        Si True, no modifica el DataFrame original.
        Si False, añade las columnas directamente sobre df.

    Retorna
    -------
    pandas.DataFrame
        DataFrame con columnas adicionales:

            parallax_corr
            distance_pc

            l_deg
            b_deg

            pm_l_cosb
            pm_b

            mu_l_solar_reflex
            mu_b_solar_reflex

            pm_l_cosb_corr
            pm_b_corr

            pmra_corr
            pmdec_corr

            d_pmra_corr
            d_pmdec_corr

            pm_corr_valid

    Uso típico
    ----------
    df_corr = correct_gaia_proper_motions(df)

    Luego usas:

        pmra_col  = "pmra_corr"
        pmdec_col = "pmdec_corr"

    """

    # ------------------------------------------------------------------
    # 1. Verificar que el DataFrame tiene las columnas necesarias
    # ------------------------------------------------------------------

    required = [ra_col, dec_col, parallax_col, pmra_col, pmdec_col]
    missing = set(required).difference(df.columns)

    if missing:
        raise ValueError(f"Faltan columnas requeridas: {sorted(missing)}")

    # Si copy=True, trabajamos sobre una copia para no alterar el DataFrame original.
    # Si copy=False, añadimos columnas directamente sobre df.
    result = df.copy() if copy else df

    # Convertimos columnas a arreglos numpy de tipo float.
    # Esto acelera las operaciones y evita problemas con tipos mixtos.
    ra = result[ra_col].to_numpy(dtype=float)
    dec = result[dec_col].to_numpy(dtype=float)
    parallax_obs = result[parallax_col].to_numpy(dtype=float)
    pmra = result[pmra_col].to_numpy(dtype=float)
    pmdec = result[pmdec_col].to_numpy(dtype=float)

    parallax_corr = parallax_obs

    # ------------------------------------------------------------------
    # 3. Máscara de estrellas válidas
    # ------------------------------------------------------------------
    #
    # Solo podemos corregir estrellas con:
    #   - ra, dec finitos
    #   - parallax_corr finita y positiva
    #   - pmra, pmdec finitos
    #
    # Si parallax <= 0, la distancia 1000/parallax no tiene sentido físico
    # en este esquema simple.
    # ------------------------------------------------------------------

    valid = (
        np.isfinite(ra)
        & np.isfinite(dec)
        & np.isfinite(parallax_corr)
        & np.isfinite(pmra)
        & np.isfinite(pmdec)
        & (parallax_corr > 0.0)
    )

    if valid.sum() == 0:
        raise ValueError("No hay estrellas válidas con parallax > 0 y PM finitos.")

    n = len(result)

    # ------------------------------------------------------------------
    # 4. Crear arreglos de salida llenos inicialmente con NaN
    # ------------------------------------------------------------------
    #
    # Para las estrellas no válidas se dejarán NaN.
    # Para las válidas se rellenarán los valores corregidos.
    # ------------------------------------------------------------------

    distance_pc = np.full(n, np.nan)

    l_deg = np.full(n, np.nan)
    b_deg = np.full(n, np.nan)

    pm_l_cosb = np.full(n, np.nan)
    pm_b = np.full(n, np.nan)

    pm_l_cosb_corr = np.full(n, np.nan)
    pm_b_corr = np.full(n, np.nan)

    pmra_corr = np.full(n, np.nan)
    pmdec_corr = np.full(n, np.nan)

    mu_l_solar = np.full(n, np.nan)
    mu_b_solar = np.full(n, np.nan)

    # ------------------------------------------------------------------
    # 5. Calcular distancia en pc
    # ------------------------------------------------------------------

    distance_pc_valid = 1000.0 / parallax_corr[valid]
    distance_pc[valid] = distance_pc_valid

    # ------------------------------------------------------------------
    # 6. Crear objeto SkyCoord en coordenadas ICRS
    # ------------------------------------------------------------------
    #
    # Gaia entrega:
    #     ra, dec, pmra, pmdec
    #
    # en el sistema ecuatorial ICRS.

    c_icrs = SkyCoord(
        ra=ra[valid] * u.deg,
        dec=dec[valid] * u.deg,
        distance=distance_pc_valid * u.pc,
        pm_ra_cosdec=pmra[valid] * u.mas / u.yr,
        pm_dec=pmdec[valid] * u.mas / u.yr,
        radial_velocity=np.zeros(valid.sum()) * u.km / u.s,
        frame="icrs",
    )

    # ------------------------------------------------------------------
    # 7. Transformar a coordenadas galácticas
    # ------------------------------------------------------------------
    #
    # En coordenadas galácticas es más natural aplicar:
    #   - corrección por movimiento solar reflejo
    #   - corrección por rotación galáctica diferencial
    #
    # Astropy devuelve:
    #     pm_l_cosb = mu_l* = mu_l cos(b)
    #     pm_b      = mu_b
    #
    # en mas/yr.
    # ------------------------------------------------------------------

    c_gal = c_icrs.galactic

    l = c_gal.l.to_value(u.rad)
    b = c_gal.b.to_value(u.rad)

    l_deg[valid] = c_gal.l.to_value(u.deg)
    b_deg[valid] = c_gal.b.to_value(u.deg)

    pm_l_cosb_valid = c_gal.pm_l_cosb.to_value(u.mas / u.yr)
    pm_b_valid = c_gal.pm_b.to_value(u.mas / u.yr)

    pm_l_cosb[valid] = pm_l_cosb_valid
    pm_b[valid] = pm_b_valid

    # Estas son las variables que iremos corrigiendo.
    # Empezamos desde los movimientos propios observados en coordenadas galácticas.
    pm_l_new = pm_l_cosb_valid.copy()
    pm_b_new = pm_b_valid.copy()

    # ------------------------------------------------------------------
    # 8. Corrección por movimiento solar reflejo
    # ------------------------------------------------------------------
    #
    # El Sol se mueve respecto al LSR con velocidad (U, V, W).
    # Esa velocidad produce un movimiento aparente reflejo en las estrellas.
    #
    # Para una estrella que estuviera en reposo respecto al LSR, nosotros
    # observaríamos un movimiento propio aparente causado por el movimiento
    # del Sol.
    #
    # En coordenadas galácticas, la contribución reflejo es:
    #
    #   mu_l_solar =
    #       parallax / KAPPA * ( U sin(l) - V cos(l) )
    #
    #   mu_b_solar =
    #       parallax / KAPPA *
    #       ( U cos(l) sin(b) + V sin(l) sin(b) - W cos(b) )
    #
    # donde:
    #     parallax está en mas
    #     U,V,W están en km/s
    #     mu queda en mas/yr
    #
    # Para obtener el movimiento propio corregido al LSR, restamos esta
    # contribución:
    #
    #     mu_corr = mu_obs - mu_solar_reflex
    #
    # ------------------------------------------------------------------

    if correct_solar_reflex:
        U, V, W = solar_motion_uvw_kms

        mu_l_solar_valid = (
            parallax_corr[valid]
            / KAPPA
            * (U * np.sin(l) - V * np.cos(l))
        )

        mu_b_solar_valid = (
            parallax_corr[valid]
            / KAPPA
            * (
                U * np.cos(l) * np.sin(b)
                + V * np.sin(l) * np.sin(b)
                - W * np.cos(b)
            )
        )

        # Restamos el movimiento solar reflejo.
        pm_l_new = pm_l_new - mu_l_solar_valid
        pm_b_new = pm_b_new - mu_b_solar_valid

        # Guardamos las correcciones aplicadas para inspección posterior.
        mu_l_solar[valid] = mu_l_solar_valid
        mu_b_solar[valid] = mu_b_solar_valid

    # Guardamos movimientos propios corregidos en coordenadas galácticas.
    pm_l_cosb_corr[valid] = pm_l_new
    pm_b_corr[valid] = pm_b_new

    # ------------------------------------------------------------------
    # 10. Transformar los movimientos propios corregidos de vuelta a ICRS
    # ------------------------------------------------------------------
    #
    # Tu código de apex probablemente trabaja con:
    #     ra, dec, pmra, pmdec
    #
    # Por eso, después de corregir en coordenadas galácticas, volvemos a ICRS.
    #
    # El resultado final será:
    #     pmra_corr
    #     pmdec_corr
    #
    # con las mismas unidades y convención que Gaia:
    #     pmra_corr = mu_alpha* = mu_alpha cos(dec)
    # ------------------------------------------------------------------

    c_gal_corr = SkyCoord(
        l=c_gal.l,
        b=c_gal.b,
        distance=distance_pc_valid * u.pc,
        pm_l_cosb=pm_l_new * u.mas / u.yr,
        pm_b=pm_b_new * u.mas / u.yr,
        radial_velocity=np.zeros(valid.sum()) * u.km / u.s,
        frame="galactic",
    )

    c_icrs_corr = c_gal_corr.icrs

    pmra_corr[valid] = c_icrs_corr.pm_ra_cosdec.to_value(u.mas / u.yr)
    pmdec_corr[valid] = c_icrs_corr.pm_dec.to_value(u.mas / u.yr)

    # ------------------------------------------------------------------
    # 11. Añadir columnas al DataFrame
    # ------------------------------------------------------------------
    #
    # Además de las columnas finales corregidas, guardamos columnas
    # intermedias útiles para depurar, graficar y entender qué tan grande
    # fue cada corrección.
    # ------------------------------------------------------------------

    result["parallax_corr"] = parallax_corr
    result["distance_pc"] = distance_pc

    result["l_deg"] = l_deg
    result["b_deg"] = b_deg

    # Movimientos propios originales, pero expresados en coordenadas galácticas.
    result["pm_l_cosb"] = pm_l_cosb
    result["pm_b"] = pm_b

    # Contribución del movimiento solar reflejo.
    result["mu_l_solar_reflex"] = mu_l_solar
    result["mu_b_solar_reflex"] = mu_b_solar
    
    # Movimientos propios corregidos en coordenadas galácticas.
    result["pm_l_cosb_corr"] = pm_l_cosb_corr
    result["pm_b_corr"] = pm_b_corr

    # Movimientos propios corregidos en coordenadas ecuatoriales ICRS.
    # Estas son las columnas principales para tu código de apex/antapex.
    result[f"{pmra_col}{suffix}"] = pmra_corr
    result[f"{pmdec_col}{suffix}"] = pmdec_corr

    # Diferencia entre el movimiento propio corregido y el observado.
    # Sirve para ver cuánto cambió cada estrella.
    result[f"d_{pmra_col}{suffix}"] = result[f"{pmra_col}{suffix}"] - result[pmra_col]
    result[f"d_{pmdec_col}{suffix}"] = result[f"{pmdec_col}{suffix}"] - result[pmdec_col]

    # Bandera booleana:
    # True  -> la estrella tenía datos válidos y fue corregida.
    # False -> la estrella tenía datos inválidos y queda con NaN en las columnas corregidas.
    result["pm_corr_valid"] = valid

    return result

In [7]:
df

,Unnamed: 0,source_id,ra,dec,parallax,pmra,pmdec,ruwe,phot_g_mean_mag,bp_rp,...,pmra_true,pmdec_true,pmra_log_error,pmdec_log_error,pmra_error_masyr,pmdec_error_masyr,pmra_noise_masyr,pmdec_noise_masyr,pm_total_true_masyr,pm_total_observed_masyr
0,1628264,45198178534988672,62.490400,15.416884,20.7,58.510596,-9.918227,1.516279,14.580980,3.142399,...,58.510596,-9.918227,-1.394140,-0.685852,0.040352,0.206133,0.0,0.0,59.345270,59.345270
1,1628267,45293526808851200,62.865737,15.992065,20.7,57.913436,-10.942595,1.130458,13.776278,2.822394,...,57.913436,-10.942595,-0.199864,-0.496567,0.631155,0.318738,0.0,0.0,58.938157,58.938157
2,1628297,45845378567048320,64.006596,16.982964,20.7,56.083185,-12.861561,1.218000,13.754810,2.825718,...,56.083185,-12.861561,-1.042336,-0.443527,0.090712,0.360141,0.0,0.0,57.539060,57.539060
3,1629424,47620265212420096,64.508186,18.256666,20.7,55.271418,-15.073248,1.935553,7.371560,0.841778,...,55.271418,-15.073248,-0.657144,-0.175678,0.220220,0.667301,0.0,0.0,57.289898,57.289898
4,1629644,48455894049191424,63.624805,18.729881,20.7,56.698194,-15.563039,3.769246,13.204377,2.762054,...,56.698194,-15.563039,-0.852750,-0.187794,0.140362,0.648942,0.0,0.0,58.795352,58.795352
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450,2195024,3411811317161811328,72.569882,20.625627,20.7,41.687147,-21.308196,1.256963,13.134485,2.683378,...,41.687147,-21.308196,-0.833644,-0.814456,0.146675,0.153301,0.0,0.0,46.817277,46.817277
451,2196160,3409548556593300480,69.937873,18.094693,20.7,46.226208,-16.305310,1.081794,16.200205,3.600326,...,46.226208,-16.305310,-1.004504,-1.131975,0.098968,0.073795,0.0,0.0,49.017603,49.017603
452,2196198,3410559832411244160,69.965462,19.659390,20.7,46.179119,-18.970188,16.270372,14.677451,3.234291,...,46.179119,-18.970188,-1.529485,-0.873253,0.029547,0.133890,0.0,0.0,49.923732,49.923732
453,2197417,3411727788638518144,73.723012,20.869290,20.7,39.670224,-22.019055,1.011372,16.923727,3.515862,...,39.670224,-22.019055,-1.289456,-1.716822,0.051350,0.019195,0.0,0.0,45.371417,45.371417


In [8]:
df_corr = correct_gaia_proper_motions(
    df,
    ra_col="ra",
    dec_col="dec",
    parallax_col="parallax_corrected",
    pmra_col="pmra",
    pmdec_col="pmdec",
    correct_solar_reflex=True
)

In [9]:
df_corr

,Unnamed: 0,source_id,ra,dec,parallax,pmra,pmdec,ruwe,phot_g_mean_mag,bp_rp,...,pm_b,mu_l_solar_reflex,mu_b_solar_reflex,pm_l_cosb_corr,pm_b_corr,pmra_corr,pmdec_corr,d_pmra_corr,d_pmdec_corr,pm_corr_valid
0,1628264,45198178534988672,62.490400,15.416884,20.7,58.510596,-9.918227,1.516279,14.580980,3.142399,...,37.521406,56.869865,-8.674732,-10.891553,46.196139,27.609863,38.605759,-30.900733,48.523986,True
1,1628267,45293526808851200,62.865737,15.992065,20.7,57.913436,-10.942595,1.130458,13.776278,2.822394,...,36.404696,56.373617,-9.298317,-10.022733,45.703014,27.816228,37.622840,-30.097208,48.565435,True
2,1628297,45845378567048320,64.006596,16.982964,20.7,56.083185,-12.861561,1.218000,13.754810,2.825718,...,34.015488,58.138219,-11.075588,-11.730354,45.091076,26.439771,38.363327,-29.643414,51.224888,True
3,1629424,47620265212420096,64.508186,18.256666,20.7,55.271418,-15.073248,1.935553,7.371560,0.841778,...,31.851725,58.582438,-12.510616,-10.963114,44.362341,26.309259,37.363485,-28.962158,52.436733,True
4,1629644,48455894049191424,63.624805,18.729881,20.7,56.698194,-15.563039,3.769246,13.204377,2.762054,...,32.043785,56.934452,-12.072145,-7.638516,44.115930,27.902554,35.014422,-28.795641,50.577461,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450,2195024,3411811317161811328,72.569882,20.625627,20.7,41.687147,-21.308196,1.256963,13.134485,2.683378,...,20.267788,57.175083,-19.239888,-14.972308,39.507675,22.370643,35.841048,-19.316504,57.149244,True
451,2196160,3409548556593300480,69.937873,18.094693,20.7,46.226208,-16.305310,1.081794,16.200205,3.600326,...,26.388537,57.392703,-15.652208,-16.084464,42.040745,23.226290,38.557408,-22.999919,54.862717,True
452,2196198,3410559832411244160,69.965462,19.659390,20.7,46.179119,-18.970188,16.270372,14.677451,3.234291,...,24.417397,61.655985,-17.992009,-18.110954,42.409405,21.996088,40.530685,-24.183031,59.500873,True
453,2197417,3411727788638518144,73.723012,20.869290,20.7,39.670224,-22.019055,1.011372,16.923727,3.515862,...,18.613221,37.907930,-13.459450,3.469763,32.072671,27.775126,16.408469,-11.895098,38.427523,True


In [11]:
# df_corr.to_csv(project_root / "data" / "datos_resultados_modularizado" / "datos_clusterizados_todos_5d_f00_corr.csv", index=False)
df_corr.to_csv(project_root / "data" / "datos_simulados" / "mock_open_cluster_gaia_hyades_clean_corr.csv", index=False)

In [34]:
# Validación y estadísticas de las correcciones aplicadas
print(f"Total de estrellas procesadas: {len(df):,}")
print(f"\nRangos de parámetros de rotación:")
print(f"  wx: [{df['wx'].min():.1f}, {df['wx'].max():.1f}] (media: {df['wx'].mean():.2f})")
print(f"  wy: [{df['wy'].min():.1f}, {df['wy'].max():.1f}] (media: {df['wy'].mean():.2f})")
print(f"  wz: [{df['wz'].min():.1f}, {df['wz'].max():.1f}] (media: {df['wz'].mean():.2f})")

print(f"\nCorrecciones de movimiento propio (mas/yr):")
print(f"  pmra_delta: media = {(df['pmra_corrected'] - df['pmra']).mean():.4f}, "
      f"std = {(df['pmra_corrected'] - df['pmra']).std():.4f}")
print(f"  pmdec_delta: media = {(df['pmdec_corrected'] - df['pmdec']).mean():.4f}, "
      f"std = {(df['pmdec_corrected'] - df['pmdec']).std():.4f}")

print(f"\nCorrelación entre componentes de movimiento propio original:")
print(f"  r(pmra, pmdec) = {np.corrcoef(df['pmra'], df['pmdec'])[0,1]:.4f}")
print(f"\nCorrelación entre componentes de movimiento propio corregidas:")
print(f"  r(pmra_corr, pmdec_corr) = {np.corrcoef(df['pmra_corrected'], df['pmdec_corrected'])[0,1]:.4f}")


Total de estrellas procesadas: 8,167,931

Rangos de parámetros de rotación:
  wx: [-25.0, 0.0] (media: -1.13)
  wy: [-15.0, 0.0] (media: -0.66)
  wz: [0.0, 5.0] (media: 0.22)

Correcciones de movimiento propio (mas/yr):
  pmra_delta: media = 0.0296, std = 4.1328
  pmdec_delta: media = -0.2130, std = 2.5190

Correlación entre componentes de movimiento propio original:
  r(pmra, pmdec) = -0.0159

Correlación entre componentes de movimiento propio corregidas:
  r(pmra_corr, pmdec_corr) = -0.0166


In [35]:
# Benchmark: Comparación de eficiencia con la solución anterior
import time

# Crear dataframe de prueba con subset de datos para benchmark
sample_size = min(10000, len(df))
df_sample = df.iloc[:sample_size].copy()

# Método anterior (ineficiente con apply + lambda)
def benchmark_apply_method():
    def get_rotation(G):
        if G < 9:
            return -5, -3, 0
        elif G < 11:
            return -10, -5, 2
        elif G < 13:
            return -25, -15, 5
        else:
            return 0, 0, 0
    
    df_test = df_sample.copy()
    start = time.perf_counter()
    df_test[['wx', 'wy', 'wz']] = df_test['phot_g_mean_mag'].apply(lambda G: pd.Series(get_rotation(G)))
    t1 = time.perf_counter() - start
    return t1

# vectorizado
def benchmark_vectorized_method():
    start = time.perf_counter()
    G = df_sample['phot_g_mean_mag'].values
    wx, wy, wz = get_rotation_vectorized(G)
    t2 = time.perf_counter() - start
    return t2

t_apply = benchmark_apply_method()
t_vect = benchmark_vectorized_method()

print(f"Benchmark ({sample_size} filas):")
print(f"  Método con apply():     {t_apply*1000:.2f} ms")
print(f"  Método vectorizado:     {t_vect*1000:.2f} ms")
print(f"  Speedup:                {t_apply/t_vect:.1f}x más rápido")
print(f"\nEn dataset completo ({len(df):,} filas):")
print(f"  Estimado (vectorizado): {(t_vect * len(df) / sample_size)*1000:.0f} ms")


Benchmark (10000 filas):
  Método con apply():     3721.92 ms
  Método vectorizado:     1.01 ms
  Speedup:                3671.3x más rápido

En dataset completo (8,167,931 filas):
  Estimado (vectorizado): 828 ms


In [36]:
print(len(df),len(df[df['fidelity_v2']>0.5]))

8167931 40121


In [17]:
df[df['parallax']>10].drop(columns=['fidelity_v1','wx','wy','wz','parallax_corrected','pmra_corrected','pmdec_corrected']).to_csv(r'C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_shell\dataset_100pc.csv')

In [38]:
df.reset_index(drop=True).to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_shell\gaia_parallax_250_fidelity.csv")

In [37]:
df.reset_index(drop=True)

,source_id,ra,dec,parallax,pmra,pmdec,ruwe,phot_g_mean_mag,bp_rp,radial_velocity,...,astrometric_params_solved,zp,parallax_corrected,fidelity_v2,fidelity_v1,wx,wy,wz,pmra_corrected,pmdec_corrected
0,138832313879044096,46.076795,35.864350,6.694639,6.033624,-27.336710,1.043566,15.661435,2.753020,-13.066293,...,31,-0.054061,6.694693,NaN,NaN,0.0,0.0,0.0,6.033624,-27.336710
1,138944429705118336,46.736132,36.147998,4.627168,45.396808,-19.805744,1.031152,14.599747,1.872160,26.450920,...,31,-0.043037,4.627211,NaN,NaN,0.0,0.0,0.0,45.396808,-19.805744
2,138969821551607552,46.725418,36.475977,10.614968,84.123814,-46.586764,1.233279,13.714395,1.990878,53.810375,...,31,-0.043869,10.615012,NaN,NaN,0.0,0.0,0.0,84.123814,-46.586764
3,139005624398868864,46.089722,36.526846,4.764534,-8.688261,-21.951382,0.904957,18.244907,3.059847,NaN,...,95,-0.047444,4.764581,NaN,NaN,0.0,0.0,0.0,-8.688261,-21.951382
4,139012835647884416,46.002892,36.718485,6.921768,14.016314,-33.248599,1.020420,17.206024,3.051111,NaN,...,31,-0.049787,6.921818,NaN,NaN,0.0,0.0,0.0,14.016314,-33.248599
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8167926,6778550143712867328,313.895072,-36.122071,11.713349,49.728996,-1.753296,1.004384,14.669379,2.390958,-4.145230,...,31,-0.049286,11.713398,NaN,NaN,0.0,0.0,0.0,49.728996,-1.753296
8167927,6778613739292053760,313.538468,-35.962284,5.022365,9.050341,-11.870507,3.939765,19.540274,0.991568,NaN,...,31,-0.013616,5.022379,NaN,NaN,0.0,0.0,0.0,9.050341,-11.870507
8167928,6778644285098195968,313.980695,-36.077103,5.904595,62.126071,-42.779672,1.002695,19.517890,3.103392,NaN,...,95,-0.001742,5.904596,NaN,NaN,0.0,0.0,0.0,62.126071,-42.779672
8167929,6778674079288447360,314.513379,-35.823604,4.661621,-25.567933,16.410468,0.854568,11.397521,0.847132,29.053432,...,31,-0.037775,4.661659,NaN,NaN,-25.0,-15.0,5.0,2.775373,16.354244
